# Final Training & Predictions

**Objective:** Train best models on FULL training data and generate test predictions

**Workflow:**
1. Load tuning results (best hyperparameters)
2. Load training and test data
3. Train best model(s) on full training data
4. Make predictions on test set
5. Save predictions for submission/ensemble

**Note:** This is where we move from evaluation to final predictions

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
import time
import pickle
import os

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

warnings.filterwarnings('ignore')

print('✓ All libraries imported successfully')

✓ All libraries imported successfully


## 2. Load Data

In [18]:
# Update paths if needed
train_path = '/home/reu24mandaloju/projects/shell_ai_hack/data/train.csv'
test_path = '/home/reu24mandaloju/projects/shell_ai_hack/data/test.csv'
output_dir = '/home/reu24mandaloju/projects/shell_ai_hack/outputs/predictions/'

os.makedirs(output_dir, exist_ok=True)

# Load data
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Identify columns
blend_cols = [col for col in train_df.columns if 'fraction' in col]
component_prop_cols = [col for col in train_df.columns if 'Component' in col and 'Property' in col]
target_cols = [col for col in train_df.columns if 'BlendProperty' in col]

feature_cols = blend_cols + component_prop_cols

# Prepare data
X_train = train_df[feature_cols].values
y_train = train_df[target_cols].values
X_test = test_df[feature_cols].values

print(f'✓ Data loaded successfully')
print(f'  Training: {X_train.shape}')
print(f'  Test: {X_test.shape}')
print(f'  Targets: {len(target_cols)}')

✓ Data loaded successfully
  Training: (2000, 55)
  Test: (500, 55)
  Targets: 10


## 3. Load Tuning Results

In [11]:
# Load tuning results
data_dir = '/home/reu24mandaloju/projects/shell_ai_hack/data/'

catboost_df = pd.read_csv(f'{data_dir}catboost_best_params.csv')
gb_df = pd.read_csv(f'{data_dir}gb_best_params.csv')
lightgbm_df = pd.read_csv(f'{data_dir}lightgbm_best_params.csv')
rf_df = pd.read_csv(f'{data_dir}rf_best_params.csv')
xgb_df = pd.read_csv(f'{data_dir}xgb_best_params.csv')

print('✓ All tuning results loaded')

# Identify best algorithm overall
overall_mapes = {
    'CatBoost': catboost_df['Best_MAPE'].mean(),
    'Gradient Boosting': gb_df['Best_MAPE'].mean(),
    'LightGBM': lightgbm_df['Best_MAPE'].mean(),
    'Random Forest': rf_df['Best_MAPE'].mean(),
    'XGBoost': xgb_df['Best_MAPE'].mean()
}

best_algo = min(overall_mapes, key=overall_mapes.get)
print(f'\n✓ Best Algorithm (by tuned MAPE): {best_algo}')
print(f'  Average MAPE: {overall_mapes[best_algo]:.4f}')

# Top 3 for ensemble
top_3_algos = sorted(overall_mapes.items(), key=lambda x: x[1])[:3]
print(f'\n✓ Top 3 Algorithms (for ensemble):')
for i, (algo, mape) in enumerate(top_3_algos, 1):
    print(f'  {i}. {algo:20s} - MAPE: {mape:.4f}')

✓ All tuning results loaded

✓ Best Algorithm (by tuned MAPE): CatBoost
  Average MAPE: 0.7872

✓ Top 3 Algorithms (for ensemble):
  1. CatBoost             - MAPE: 0.7872
  2. XGBoost              - MAPE: 0.8350
  3. Gradient Boosting    - MAPE: 0.9215


## 4. Train Best Algorithm on Full Data

In [12]:
print('\n' + '='*80)
print(f'TRAINING: {best_algo.upper()} ON FULL TRAINING DATA')
print('='*80)

# Select appropriate results dataframe
if best_algo == 'CatBoost':
    best_params_df = catboost_df
elif best_algo == 'Gradient Boosting':
    best_params_df = gb_df
elif best_algo == 'LightGBM':
    best_params_df = lightgbm_df
elif best_algo == 'Random Forest':
    best_params_df = rf_df
else:  # XGBoost
    best_params_df = xgb_df

# Train model for each target
best_models = {}

print(f'\nTraining {best_algo} for each target on full training data (2000 samples)...\n')

for idx, target_name in enumerate(target_cols):
    print(f'⏳ Training for {target_name}...')
    
    # Get target data
    y = y_train[:, idx]
    
    # Get best hyperparameters for this target
    params_row = best_params_df[best_params_df['Target'] == target_name].iloc[0]
    params = params_row.drop(['Target', 'Best_MAPE']).to_dict()
    
    # Create and train model
    start = time.time()
    
    if best_algo == 'CatBoost':
        model = CatBoostRegressor(
            random_state=42,
            verbose=False,
            task_type='GPU',
            **params
        )
    elif best_algo == 'Gradient Boosting':
        model = GradientBoostingRegressor(random_state=42, **params)
    elif best_algo == 'LightGBM':
        model = LGBMRegressor(
            random_state=42,
            verbose=-1,
            device_type='gpu',
            **params
        )
    elif best_algo == 'Random Forest':
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    else:  # XGBoost
        model = XGBRegressor(
            random_state=42,
            verbosity=0,
            tree_method='hist',
            **params
        )
    
    model.fit(X_train, y)
    elapsed = time.time() - start
    
    best_models[target_name] = model
    print(f'  ✓ Completed in {elapsed:.2f}s')

print(f'\n✓ All {len(best_models)} models trained successfully')


TRAINING: CATBOOST ON FULL TRAINING DATA

Training CatBoost for each target on full training data (2000 samples)...

⏳ Training for BlendProperty1...
  ✓ Completed in 5.98s
⏳ Training for BlendProperty2...
  ✓ Completed in 2.10s
⏳ Training for BlendProperty3...
  ✓ Completed in 2.07s
⏳ Training for BlendProperty4...
  ✓ Completed in 1.88s
⏳ Training for BlendProperty5...
  ✓ Completed in 2.15s
⏳ Training for BlendProperty6...
  ✓ Completed in 1.73s
⏳ Training for BlendProperty7...
  ✓ Completed in 2.72s
⏳ Training for BlendProperty8...
  ✓ Completed in 2.07s
⏳ Training for BlendProperty9...
  ✓ Completed in 2.10s
⏳ Training for BlendProperty10...
  ✓ Completed in 1.78s

✓ All 10 models trained successfully


## 5. Make Predictions on Test Set

In [16]:
print('\n' + '='*80)
print('GENERATING TEST PREDICTIONS')
print('='*80)

# Generate predictions
predictions_best = []

print(f'\nGenerating {best_algo} predictions for {len(test_df)} test samples...\n')

for target_name in target_cols:
    model = best_models[target_name]
    preds = model.predict(X_test)
    predictions_best.append(preds)
    print(f'  ✓ {target_name} predictions generated')

# Convert to proper format (500 rows, 10 columns)
predictions_best_array = np.array(predictions_best).T
submission_best = pd.DataFrame(predictions_best_array, columns=target_cols)

print(f'\n✓ Predictions generated')
print(f'  Shape: {submission_best.shape}')
print(f'  Columns: {list(submission_best.columns)}')
print(f'\n  First 5 predictions:')
print(submission_best.head())


GENERATING TEST PREDICTIONS

Generating CatBoost predictions for 500 test samples...

  ✓ BlendProperty1 predictions generated
  ✓ BlendProperty2 predictions generated
  ✓ BlendProperty3 predictions generated
  ✓ BlendProperty4 predictions generated
  ✓ BlendProperty5 predictions generated
  ✓ BlendProperty6 predictions generated
  ✓ BlendProperty7 predictions generated
  ✓ BlendProperty8 predictions generated
  ✓ BlendProperty9 predictions generated
  ✓ BlendProperty10 predictions generated

✓ Predictions generated
  Shape: (500, 10)
  Columns: ['BlendProperty1', 'BlendProperty2', 'BlendProperty3', 'BlendProperty4', 'BlendProperty5', 'BlendProperty6', 'BlendProperty7', 'BlendProperty8', 'BlendProperty9', 'BlendProperty10']

  First 5 predictions:
   BlendProperty1  BlendProperty2  BlendProperty3  BlendProperty4  \
0        0.105553        0.188169        0.660775        0.537461   
1       -0.682798       -0.675303       -1.215897       -0.026789   
2        1.690270        1.063956 

## 6. Save Predictions

In [19]:
# Save best algorithm predictions
output_path_best = f'{output_dir}predictions_{best_algo.replace(" ", "_").lower()}.csv'
submission_best.to_csv(output_path_best, index=False)

print(f'✓ Saved {best_algo} predictions to: {output_path_best}')

# Save models for later ensemble use
models_path = f'{output_dir}models_{best_algo.replace(" ", "_").lower()}.pkl'
with open(models_path, 'wb') as f:
    pickle.dump(best_models, f)

print(f'✓ Saved trained models to: {models_path}')

✓ Saved CatBoost predictions to: /home/reu24mandaloju/projects/shell_ai_hack/outputs/predictions/predictions_catboost.csv
✓ Saved trained models to: /home/reu24mandaloju/projects/shell_ai_hack/outputs/predictions/models_catboost.pkl


## 7. Train Top 3 Algorithms (for Ensemble)

In [20]:
print('\n' + '='*80)
print('TRAINING TOP 3 ALGORITHMS (FOR ENSEMBLE)')
print('='*80)

# Mapping of algorithm names to dataframes
algo_mapping = {
    'CatBoost': catboost_df,
    'Gradient Boosting': gb_df,
    'LightGBM': lightgbm_df,
    'Random Forest': rf_df,
    'XGBoost': xgb_df
}

all_predictions = {}  # Store predictions from all top 3
all_models = {}       # Store models from all top 3

for rank, (algo_name, _) in enumerate(top_3_algos, 1):
    print(f'\n{rank}. Training {algo_name}...')
    
    best_params_df = algo_mapping[algo_name]
    algo_models = {}
    algo_predictions = []
    
    for idx, target_name in enumerate(target_cols):
        y = y_train[:, idx]
        
        params_row = best_params_df[best_params_df['Target'] == target_name].iloc[0]
        params = params_row.drop(['Target', 'Best_MAPE']).to_dict()
        
        if algo_name == 'CatBoost':
            model = CatBoostRegressor(random_state=42, verbose=False, task_type='GPU', **params)
        elif algo_name == 'Gradient Boosting':
            model = GradientBoostingRegressor(random_state=42, **params)
        elif algo_name == 'LightGBM':
            model = LGBMRegressor(random_state=42, verbose=-1, device_type='gpu', **params)
        elif algo_name == 'Random Forest':
            model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        else:  # XGBoost
            model = XGBRegressor(random_state=42, verbosity=0, tree_method='hist', **params)
        
        model.fit(X_train, y)
        algo_models[target_name] = model
        
        # Predict on test set
        preds = model.predict(X_test)
        algo_predictions.append(preds)
    
    # Store results
    preds_array = np.array(algo_predictions).T
    all_predictions[algo_name] = pd.DataFrame(preds_array, columns=target_cols)
    all_models[algo_name] = algo_models
    
    print(f'  ✓ {algo_name} training complete')

print(f'\n✓ All top 3 algorithms trained successfully')


TRAINING TOP 3 ALGORITHMS (FOR ENSEMBLE)

1. Training CatBoost...
  ✓ CatBoost training complete

2. Training XGBoost...
  ✓ XGBoost training complete

3. Training Gradient Boosting...
  ✓ Gradient Boosting training complete

✓ All top 3 algorithms trained successfully


## 8. Save Top 3 Predictions

In [21]:
print('\nSaving Top 3 predictions for ensemble...\n')

for algo_name, preds_df in all_predictions.items():
    output_path = f'{output_dir}predictions_{algo_name.replace(" ", "_").lower()}.csv'
    preds_df.to_csv(output_path, index=False)
    print(f'  ✓ Saved {algo_name} predictions')
    
    # Also save models
    models_path = f'{output_dir}models_{algo_name.replace(" ", "_").lower()}.pkl'
    with open(models_path, 'wb') as f:
        pickle.dump(all_models[algo_name], f)
    print(f'    Saved {algo_name} models')

print(f'\n✓ All predictions and models saved')


Saving Top 3 predictions for ensemble...

  ✓ Saved CatBoost predictions
    Saved CatBoost models
  ✓ Saved XGBoost predictions
    Saved XGBoost models
  ✓ Saved Gradient Boosting predictions
    Saved Gradient Boosting models

✓ All predictions and models saved


## 9. Summary & Next Steps

In [22]:
print('\n' + '='*80)
print('FINAL TRAINING COMPLETE')
print('='*80)

print(f'\n✓ TRAINED MODELS:')
print(f'  Best Algorithm: {best_algo}')
print(f'  Top 3 Algorithms: {[algo for algo, _ in top_3_algos]}')

print(f'\n✓ SAVED FILES:')
print(f'  Best predictions: predictions_{best_algo.replace(" ", "_").lower()}.csv')
for algo_name in [algo for algo, _ in top_3_algos]:
    print(f'  {algo_name} predictions: predictions_{algo_name.replace(" ", "_").lower()}.csv')

print(f'\n  Trained models saved for ensemble use')

print(f'\n📋 WHAT TO DO NEXT:')
print(f'  Option 1: Use {best_algo} predictions directly for submission')
print(f'  Option 2: Run 06_ensemble_methods.ipynb to combine top 3')
print(f'           (usually improves score by 5-15%)')

print(f'\n' + '='*80)


FINAL TRAINING COMPLETE

✓ TRAINED MODELS:
  Best Algorithm: CatBoost
  Top 3 Algorithms: ['CatBoost', 'XGBoost', 'Gradient Boosting']

✓ SAVED FILES:
  Best predictions: predictions_catboost.csv
  CatBoost predictions: predictions_catboost.csv
  XGBoost predictions: predictions_xgboost.csv
  Gradient Boosting predictions: predictions_gradient_boosting.csv

  Trained models saved for ensemble use

📋 WHAT TO DO NEXT:
  Option 1: Use CatBoost predictions directly for submission
  Option 2: Run 06_ensemble_methods.ipynb to combine top 3
           (usually improves score by 5-15%)

